# FIFA World Cup 2026 - Baseline Predictor

Pipeline:
1. Download ~150 years of international match results.
2. Compute World Football Elo ratings over the full history.
3. Fit a multinomial logit mapping Elo gap -> (Home, Draw, Away) probabilities.
4. Monte-Carlo simulate the 48-team tournament 10k times.
5. Read off P(champion) per team.

Edit `src/groups_2026.py` with the actual draw before trusting the numbers.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import matplotlib.pyplot as plt

from src.data import load_results
from src.elo import compute_elo_history, top_n
from src.model import fit_outcome_model, match_proba
from src.simulate import monte_carlo
from src.groups_2026 import GROUPS_2026

## 1. Load match history

In [ ]:
results = load_results()
print(f'{len(results):,} matches from {results.date.min().date()} to {results.date.max().date()}')
results.tail()

## 2. Run Elo over the full history

In [ ]:
ratings, snapshots = compute_elo_history(results)
top_n(ratings, 20)

## 3. Fit outcome model (Elo diff -> H/D/A probability)

In [ ]:
model = fit_outcome_model(snapshots, min_date='2006-01-01')

diffs = np.linspace(-400, 400, 81)
probs = model.proba(diffs)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(diffs, probs[:, 0], label='P(Home win)')
ax.plot(diffs, probs[:, 1], label='P(Draw)')
ax.plot(diffs, probs[:, 2], label='P(Away win)')
ax.set_xlabel('Elo diff (home - away)')
ax.set_ylabel('Probability')
ax.legend(); ax.grid(alpha=0.3)
plt.show()

## 4. Sanity check on individual matchups

In [ ]:
for a, b in [('Argentina', 'France'), ('Spain', 'England'), ('Brazil', 'Germany')]:
    p = match_proba(model, ratings[a], ratings[b], neutral=True)
    print(f'{a} vs {b}: H={p["H"]:.2f} D={p["D"]:.2f} A={p["A"]:.2f}')

## 5. Monte-Carlo simulate the 2026 tournament

Replace `GROUPS_2026` in `src/groups_2026.py` with the official draw first.

In [ ]:
missing = [t for grp in GROUPS_2026.values() for t in grp if t not in ratings]
if missing:
    print('WARNING: not in Elo table (typo or rare name):', missing)

predictions = monte_carlo(GROUPS_2026, ratings, model, n_sims=10_000, seed=42)
predictions.head(15)

In [ ]:
top = predictions.head(12).iloc[::-1]
fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(top['team'], top['p_champion'])
ax.set_xlabel('P(champion)')
ax.set_title('Top 12 - 2026 FIFA World Cup baseline')
plt.tight_layout(); plt.show()

## Next steps
- Add host-nation boost (USA/Canada/Mexico get +50 Elo for home games).
- Replace random R32 draw with FIFA's actual bracket once known.
- Layer in player market value / availability as covariates.
- Blend with bookmaker odds as a calibration target.
- Compare against the Opta supercomputer's top-5 as a smoke test.